In [ ]:
import glob
import os
import pandas as pd
import re
import sys

if '../' not in sys.path:
    sys.path.append('../')

from emp.dataset import extract_bl_shelfmark, gen_prompt

In [ ]:
def extract_title(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return " ".join(s.split()[:-2])
    elif date_re.search(s.split()[-1]):
        return " ".join(s.split()[:-1]) 
    elif s.split()[-1] in ["a", "b", "c"]:
        return " ".join(s.split()[:-1])
    else:
        return s

In [ ]:
def extract_edition(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return s.split()[-2]
    elif date_re.search(s.split()[-1]):
        return s.split()[-1] 
    elif s.split()[-1] in ["a", "b", "c"]:
        return s.split()[-1]
    else:
        return None

## Title Locations

#### title_loc_df

In [ ]:
title_loc_path = "../data/interim/title_loc_df.csv"
title_loc_df = pd.read_csv(title_loc_path, encoding="utf-8-sig", index_col=0)
assert title_loc_df.index.is_unique

# One addition for a title not in Proudfoot
title_loc_df.loc["Catechism - Penang", "entry_text"] = """
Catechism - Penang
1826
author/translator: not given
proprietor, publisher & printer: not given
Location
BL OIOC ORB.30/5553 
"""

In [ ]:
entry = "Pelajaran Bahasa Arab"
# gen_prompt(title_loc_df.loc[entry, "entry_text"], entry)

### Boll EMP

#### Boll EMP df

In [ ]:
# 1 Confirmed duplicate row for Miftah al-Bayan 1890
boll_emp_df = pd.read_csv("../data/external/boll_emp_by_date.csv", encoding="utf8", skipfooter=3, engine='python', usecols=[0, 1, 2, 3]).drop_duplicates(subset=["shelfmark", "title_edition"])

boll_emp_df["shelfmark"] = boll_emp_df["shelfmark"].str.strip()
boll_emp_df.set_index("shelfmark", inplace=True)
boll_emp_df.rename(index={"14625.d. 16": "14625.d.16"}, inplace=True)
assert boll_emp_df.index[-1] == "14625.e.5"
assert boll_emp_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

In [ ]:
# modifications to boll_emp_df
# 22/06/26 have double checked all these corrections are applied to the correct cells
boll_emp_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
boll_emp_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
boll_emp_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
boll_emp_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
boll_emp_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
boll_emp_df.loc["14626.c.14(4)", "title_edition"] = "Haris Fadhillah 1900"
boll_emp_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
boll_emp_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
boll_emp_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
boll_emp_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
boll_emp_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
boll_emp_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
boll_emp_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
boll_emp_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
boll_emp_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
boll_emp_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
boll_emp_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
boll_emp_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"

In [ ]:
boll_emp_df["title"] = boll_emp_df["title_edition"].apply(lambda x: extract_title(x))
boll_emp_df["edition"] = boll_emp_df["title_edition"].apply(lambda x: extract_edition(x))

In [ ]:
assert boll_emp_df.set_index(["title"]).index.difference(title_loc_df.index).empty
assert not boll_emp_df["edition"].hasnans

#### boll_entry_df

In [ ]:
boll_entry_df = title_loc_df.merge(boll_emp_df, how="inner", left_index=True, right_on="title").reset_index()
assert boll_entry_df.drop_duplicates(subset="title")["entry_start"].is_monotonic_increasing
boll_entry_df.set_index("title", inplace=True)
assert not boll_entry_df["entry_text"].hasnans

In [ ]:
boll_entry_df["extracted_shelfmark"] = boll_entry_df["entry_text"].apply(lambda x: extract_bl_shelfmark(x)).str.strip().str.replace("v. ", "v.")

In [ ]:
absent_raw_sm_df = boll_entry_df[~(boll_entry_df.apply(lambda x: x["shelfmark"] in x["entry_text"], axis=1))]
absent_sm_df = absent_raw_sm_df[absent_raw_sm_df["shelfmark"] != absent_raw_sm_df["extracted_shelfmark"]]

In [ ]:
assert len(absent_sm_df) == 11  # I've accounted for these 11

In [ ]:
absent_sm_df

In [ ]:
# checking that extract_bl_shelfmark will find the correct shelfmark
# correct shelfmark -> shelfmark in entry -> entry sm will be corrected by extract_bl_shelfmark

# 14620.k.1 -> 14620.k.l -> yes
# Jav.73 -> Jav. 73 -> yes

# 14653.d.1 -> 14653.d.l -> no  # but only because there are multiple shelfmarks for this edition, will probably have to pick this up in post

# Jav.70 -> Jav. 70 -> yes

# 14626.d.11(2) -> 14626.d.ll(2) -> yes

# 14620.c.10 -> 14620.c.1O -> yes

# ORB.30/612 -> ORB. 30/612 -> yes

# 14653.b.1 -> 14653.b.l -> yes

# 14620.d.19(14) -> 14620.d.19(l4) -> yes

# 14625.a.1 -> 14625.a.l -> yes

# 14626.d.11(6) -> 14626.d.ll(6) -> yes (unclear why not picked up already)

# fixed by modifying extract_bl_shelfmark

# 14626.c.10 -> 14626.c.1O -> yes
# 14626.a.10 -> 14626.a.1O -> yes
# 14625.e.10 -> 14625.e.1O -> yes
# 14626.d.11(10) -> 14626.d.11(1O) -> yes
# 14626.d.10 -> 14626.d.1O -> yes
# ORB.30/452 -> ORB. 30/452 -> yes

# fixed by modifying entry in extract_clean_entries

# 14620.a.19(10) -> 14620.aI9(10) -> no  # sm missing a '.', will need to add in
# 14620.b.18(10) -> 14620.b.18{l0) -> no  # will need to modify shelfmark
# 14623.c.4 -> 14623.cA -> no  # need to modify shelfmark
# 14626.d.11(8) -> 14626.d.l1 (8) -> no  # may need to modify sm to remove space
# 14625.a.9 -> 14625.a9 -> no  # may need to add missing '.'
# 14620.g.20(0) -> 14620.g.20(-) -> no  # may need to modify sm to swap -/0
# 14626.e.4 -> 14626.eA -> no  # will need to add '.' to sm and swap A/4
# 14633.a.38 -> 1463303.38 -> no  # will need to fix sm

# 14620.g.11 -> -> no  # may need to modify entry extent
# 14620.d.14 -> -> no  # may need to modify entry extend
# 14625.e.13 -> -> no  # need to modify entry extent
# 14629.c.31 -> -> no  # no, may need to modify entry
# 14620.g.19(9) ->  -> no  # may need to modify entry extent
# 14620.d.11 ->  -> no  # may need to modify entry

In [ ]:
# with open("../data/interim/sm_check.txt", "w") as f:
#     [f.write(r[1]["shelfmark"] + "\n" + r[1]["extracted_shelfmark"] + "\n\n" + r[1]["entry_text"] + "\n\n") for r in absent_sm_df.iterrows()]

In [ ]:
boll_entry_df[~(boll_entry_df.apply(lambda x: x["edition"] in x["entry_text"], axis=1))]

### Export prepared entries to interim

In [ ]:
# boll_entry_df.drop_duplicates(subset="title").to_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig", index=False)

In [ ]:
# pd.read_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig").set_index("title")

## AAC

In [ ]:
full_aac_df = pd.read_csv("../data/external/full_aac_by_date.csv", encoding="utf8", usecols=[0,1,2,3]).drop_duplicates(subset=["shelfmark", "title_edition", "year"])

In [ ]:
full_aac_df["shelfmark"] = full_aac_df["shelfmark"].apply(
    lambda x: x.split(" (IOLR")[0]
).str.strip(
).str.replace("° ", ""
).str.replace(". ", "."
).str.replace(" .", "."
).str.replace("( ", "("
).str.replace("{", "("
).str.replace(".3.", ".a."
).str.replace(".33.", ".aa.")

full_aac_df.set_index("shelfmark", inplace=True)
full_aac_df.rename(index={
    "o14633.d.10": "14633.d.10",
    "14625.e,17(1)": "14625.e.17(1)",
    "l": "14626.c.16(1)",
    "14620.g.20(-)": "14620.g.20(0)",
    "14620.f.3(I)": "14620.f.3(1)",
    "I4625.g.1": "14625.g.1"
}, inplace=True)

# Verified 30/06/26
full_aac_df.loc["14620.k.1", "title_edition"] = "Anbiya 1897"
full_aac_df.loc["14628.c.2(3)", "title_edition"] = "Awang Sulung Merah Muda 1914"
full_aac_df.loc["14620.g.11", "title_edition"] = "Bidayat al-Salikin 1906"
full_aac_df.loc["14620.d.20(1)", "title_edition"] = "Bustan Arifin 1820-1822"
full_aac_df.loc["ORB.30/5553", "title_edition"] = "Catechism - Penang"
full_aac_df.loc["14625.d.14", "title_edition"] = "Dandan Setia 1894"
full_aac_df.loc["14625.c.47", "title_edition"] = "Dermah Tasiah 1906"
full_aac_df.loc["14620.a.2", "title_edition"] = "Dini Hari 1863"
full_aac_df.loc["14620.b.33(1)", "title_edition"] = "Henry b"
full_aac_df.loc["14620.ff.1", "title_edition"] = "Hidayat al-Awamm 1903"
full_aac_df.loc["14620.g.17", "title_edition"] = "Hilyat al-Anam 1893"
full_aac_df.loc["Jav.76", "title_edition"] = "Husn al-Akhlak 1900"
full_aac_df.loc["14620.b.14(8)", "title_edition"] = "Ilmu Kepandaian 1843"
full_aac_df.loc["14629.e.2", "title_edition"] = "Ilmu Kepandaian a 1839"
full_aac_df.loc["14653.d.25", "title_edition"] = "Johor 1920"
full_aac_df.loc["14620.ee.29", "title_edition"] = "Kebaktian Sehari-harian 1905"
full_aac_df.loc["14625.c.17", "title_edition"] = "Lautan Akal 1907"
full_aac_df.loc["14654.b.59(2(2))", "title_edition"] = "Madat 1919"
full_aac_df.loc["14519.d.44(1)", "title_edition"] = "Maulud 1871.a"
full_aac_df.loc["14519.d.9(3)", "title_edition"] = "Maulud 1871.b"
full_aac_df.loc["14620.g.7", "title_edition"] = "Miftah al-Bayan 1890"
full_aac_df.loc["14620.d.30(3)", "title_edition"] = "Orang Bergantung kepada Isa a"
full_aac_df.loc["14620.d.30(1)*", "title_edition"] = "Orang Miskin a"
full_aac_df.loc["14629.a.1(1)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) 1847"
full_aac_df.loc["14620.d.17(4)", "title_edition"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
full_aac_df.loc["14653.d.20", "title_edition"] = "Penimbau Akal 1920"
full_aac_df.loc["14620.d.20(16)", "title_edition"] = "Perkataan Isa 1832"
full_aac_df.loc["14625.aa.2(1)", "title_edition"] = "Sabar Ali 1851"
full_aac_df.loc["14620.f.5", "title_edition"] = "Tahsil al-Ujur 1893"
full_aac_df.loc["14620.g.5", "title_edition"] = "Targhib al-Nas 1873"
full_aac_df.loc["14620.g.5*", "title_edition"] = "Targhib al-Nas 1873"
full_aac_df.loc["14626.a.37", "title_edition"] = "Umm al-Burhan a"
full_aac_df.loc["14622.b.15", "title_edition"] = "Undang-Undang Kapal 1916"
full_aac_df.loc["14620.c.3(3)", "title_edition"] = "Wasiat Lama 1856"
full_aac_df.loc["14625.b.1", "title_edition"] = "Yue Fei 1891"

assert full_aac_df["title_edition"].str.contains("Azimat a - fragment in binding").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Azimat a - fragment in binding", "Azimat a")

assert full_aac_df["title_edition"].str.contains("Bidasari 1903.a colophon only").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Bidasari 1903.a colophon only", "Bidasari 1903.a")

assert full_aac_df["title_edition"].str.contains("Terasul 1899 in binding only").sum() == 1
full_aac_df["title_edition"] = full_aac_df["title_edition"].str.replace("Terasul 1899 in binding only", "Terasul 1899")

In [ ]:
full_aac_df["title"] = full_aac_df["title_edition"].apply(lambda x: extract_title(x))
# apply fixes
# preceding '?' and missing spaces
full_aac_df["title"] = full_aac_df["title"].str.replace("? ", "").str.replace(
    "AbuNawas", "Abu Nawas").str.replace(
    "BarangSiapa", "Barang Siapa").str.replace(
    "CeritaJenaka", "Cerita Jenaka").str.replace(
    "KenTabuhan", "Ken Tabuhan").str.replace(
    "HajjdanUmrah", "Hajj dan Umrah").str.replace(
    "HangTuah", "Hang Tuah").str.replace(
    "MaiYoulang", "Mai Youlang").str.replace(
    "Misal HurufRumi", "Misal Huruf Rumi").str.replace(
    "NasihatBidan", "Nasihat Bidan").str.replace(
    "PenerangHati", "Penerang Hati").str.replace(
    "RajaMuda", "Raja Muda").str.replace(
    "RencanaPiatu", "Rencana Piatu")
full_aac_df["edition"] = full_aac_df["title_edition"].apply(lambda x: extract_edition(x))
full_aac_df.loc["ORB.30/5553", "edition"] = "1826"

In [ ]:
assert boll_emp_df.loc[boll_emp_df.index.difference(full_aac_df.index)].empty
assert full_aac_df.set_index(["title"]).index.difference(title_loc_df.index).empty
assert full_aac_df.reset_index().value_counts(subset=["shelfmark", "title_edition"]).max() == 1

In [ ]:
non_boll_aac_df = full_aac_df.set_index("title_edition", append=True).loc[full_aac_df.set_index("title_edition", append=True).index.difference(boll_emp_df.set_index("title_edition", append=True).index)]
assert len(non_boll_aac_df) + len(boll_emp_df) == len(full_aac_df)
len(non_boll_aac_df), len(boll_emp_df), len(full_aac_df)

In [ ]:
non_boll_aac_df.sort_values(by=["title_edition", "year"]).to_excel("../data/interim/non_boll_aac_df.xlsx")

### aac_entry_df

In [ ]:
non_boll_aac_entry_df = title_loc_df.merge(non_boll_aac_df, how="inner", left_index=True, right_on="title").reset_index()

In [ ]:
non_boll_aac_entry_df.shape

In [ ]:
# add a dropna() due to the added Catechism - Penang which only has entry_text
assert non_boll_aac_entry_df.drop_duplicates(subset="title").dropna(subset="entry_start")["entry_start"].is_monotonic_increasing
# aac_entry_df.set_index("title", inplace=True)
assert not non_boll_aac_entry_df["entry_text"].hasnans

In [ ]:
non_boll_aac_entry_df[non_boll_aac_entry_df["entry_text"].isna()]

In [ ]:
non_boll_aac_entry_df["extracted_shelfmark"] = non_boll_aac_entry_df["entry_text"].apply(
    lambda x: extract_bl_shelfmark(x)).str.strip().str.replace("v. ", "v.")

In [ ]:
absent_aac_raw_sm_df = non_boll_aac_entry_df[~(non_boll_aac_entry_df.apply(lambda x: x["shelfmark"] in x["entry_text"], axis=1))]
absent_aac_sm_df = absent_aac_raw_sm_df[absent_aac_raw_sm_df["shelfmark"] != absent_aac_raw_sm_df["extracted_shelfmark"]]

In [ ]:
assert len(absent_aac_sm_df) == 62

14625.e.11(3) , Abdul Muluk 1891
BL OIOC 14625.e.ll(3)

ORB.30/445 , Abdullah 1849
OIOC ORB. 30/445 (IOLR Malay F6
306/36.G.7 ?)

ORB.30/446 , Abdullah 1849
ORB. 30/446 (IOLR
Malay F6 306/36.G.8)

ORB.30/447 , Abdullah 1849
ORB. 30/447
(IOLR Malay F6 306/36.G.15)

ORB.30/448 , Abdullah 1849
ORB. 30/448 (IOLR Malay F6
306/36.G.16)

14628.c.1(4) , Abdullah 1907
14628.c.l(4)

14653.d.4(1,2) , Abdullah 1907 
14653.d.4(1),(2) (lOLR
Malay
02/4(1),(2 

14653.d.46(1,2) , Abdullah 1911 
BL OIOC 14653.d.46(1),(2) (IOLR
Malay 030(1),(2

14625.a.10 , AbuNawas 1917
BL OIOC 14625.a.1O

14626.d.11(7) , Air Mawar 1899
BLOIOC 14626.dl1(7)

14653.d.11 , Anggun Cik Tunggal 1914 
14653.d.ll (IOLR Malay 02/11)

14620.d.17(11) , Bible: Acts a c.1829
BL OIOC 14620.d.17(1l)

Jav.119(3) , Bible: Acts 1898
BL OIOC Jav. 119(3)

Jav.119(4) , Bible: Acts 1904.a 
BL OIOC Jav. 119(4)

14620.b.14(10) , Bible: Genesis a
BL OIOC 14620.b.14(l0

Jav.100 , Bible: Genesis 1908
BL OIOC Jav. 100

Jav.117 , Bible: Luke 1893.b 
BL OIOC Jav. 117

Jav.119(2) , Bible: Luke 1896
BL OIOC Jav. 119(2)

Jav.119(1) , Bible: Mark 1899.a 
BL OIOC Jav. 119(1)

Jav.116 , Bible: Matthew 1897.b 
BL OIOC Jav. 116

ORB.30/453 , Bible: New Testament 1856
OIOC ORB. 30/453 (IOLR Malay
306/36.H.9

Jav.121 , Bible: New Testament 1911.a 
BL OIOC Jav. 121

Jav.161 , Bible: New Testament 1911.a
BL OIOC Jav. 161

Jav.96 , Bible: New Testament 1911.a 
BL OIOC Jav. 96

Jav.102 , Bible: Proverbs 1909 
Jav. 102

Jav.101 , Bible: Psalms 1909.b 
BL OIOC Jav. 101

14620.c.19(10) , Bible: Romans 1900
OIOC
14620.c.19(lO)

14620.c.19(11) , Bible: Romans 1908
BL OIOC 14620.c.19(1l)

14623.a.1 , Binatang 1846
OIOC 14623.a.l

14620.d.17(6) , Cakrawala a c.1840s
BL OIOC 14620.d. I 7(6)

14628.c.1(6) , CeritaJenaka 1908
14628.c.l(6)

ORB.30/457 , Cermin Mata 1858
ORB. 30/457

14653.b.33(3,4,5) , Dondang Sayang 1911 
14653.b.33(3),(4),(5)

14626.a.1 , Dondang Sayang 1915
OIOC
14626.a.l

Jav.82 , Faslatan 1906 
BL OIoe Jav. 82

14653.d.3(2,3,4) , Hang Tuah 1913 
BL OIOC 14653.d.3(2),(3),(4)

14653.b.10 , Hidayat al-Muslimin 1916 
BL OIoe 14653.b.1O

14620.b.1 , Hikayat al-Kitab 1855
BL OIOC 14620.b.l

14653.d.25 , Johor 1920
BL OIOC l4653.d.25

14625.e.11(2) , Kahar Masyhur 1891
BL OIOC 14625.e.1l(2)

Jav.83 , Lataif al-Taharat 1906.b 
BL OIOC Jav. 83

Jav.63 , Majmuah al-Syariah 1894 
BL OIOC Jav. 63

Jav.65 , Majmuah al-Syariah 1906 
BL OIOC Jav. 65

14654.b.59(2(3)) , Makan Sirih 1919
 BL OIOC 14654.b.59(2(3 

ORB.30/451 , Miskin Marakarmah 1857
OIOC ORB. 30/451

14625.e.8 , Miskin Marakarmah 1915

14625.e.20(1) , Muhammad Hanafiah 1891
BL OIOC 14625.e.2O(1)

Jav.56 , Munjiyat 1893.b 
BL OIOC Jav. 56

Jav.17 , Munjiyat 1895 
BL OIOC Jav. 17

Jav.61 , Munjiyat 1901 
BL OIOC Jav. 61

Jav.71 , Munjiyat 1906 
BL OIOC Jav. 71

14626.d.11(11) , Nasihat Bapa 1904
BL OIOC 14626.d.ll(ll)

14653.d.28 , Orang yang Cari Selamat 1905 
I4653.d.28

14625.e.11(1) , ? Panji Semirang 1888.b
BL OIOC 14625.e.ll(l)

14629.a.1(1) , Pelajaran Bahasa Melayu (No.1) 1847
BL OIOC 14629.a.l

14629.a.1(2) , Pelajaran Bahasa Melayu (No.2) 1861 
BL OIOC 14629.a.l

14620.c.7 , Pelajaran Sekolah Sabat 1920

14628.c.3(1) , Pelayaran Abdullah 1913
BL OIOC 14628.c.3(I)

ORB.30/585 , Puji-Pujian 1896
BL OIOC ORB. 30/585

14620.d.10 , Puji-Pujian Methodist 1913
BL OIOC 14620.d.1O

14653.d.9(1,2) , Sejarah Melayu 1910 
14653.d.9(1),(2)

14625.aa.1(1)* , Terubuk 1873
14625.aa.l(I)*

In [ ]:
# with open("../data/interim/full_aac_sm_check.txt", "w") as f:
#     [f.write(r[1]["shelfmark"] + "\n" + r[1]["extracted_shelfmark"] + "\n\n" + r[1]["entry_text"] + "\n\n") for r in absent_aac_sm_df.sort_values(by=["title", "year"]).drop_duplicates(subset="title").iterrows()]

In [ ]:
non_boll_aac_entry_df[non_boll_aac_entry_df["entry_text"].isna()]

In [ ]:
non_boll_aac_entry_df[~(non_boll_aac_entry_df.apply(lambda x: x["edition"] in x["entry_text"], axis=1))]

### Export

In [ ]:
non_boll_aac_entry_df.info()

In [ ]:
non_boll_aac_entry_df.drop_duplicates(subset="title").to_csv("../data/interim/title_dedup_aac_emp.csv", encoding="utf-8-sig", index=False)


### Modify lines_to_concatenate

In [ ]:
with open("../data/interim/lines_to_concatenate_with_text.txt", encoding="utf8") as f:
    lines = [l.strip("\n").split("\t") for l in f.readlines()]
    bad_line_ids, bad_line_texts = [int(l[0]) for l in lines], [l[1] for l in lines]

In [ ]:
for i, idx in enumerate(bad_line_ids):
    if i > 92:
        bad_line_ids[i] += 1

In [ ]:
with open("../data/interim/lines_to_concatenate_with_text_new.txt", "w", encoding="utf8") as f:
    for idx, text in zip(bad_line_ids, bad_line_texts):
        # f.write(f"{idx}\t{text}\n")

In [ ]:
bad_line_ids

In [ ]:
assert not full_aac_df.set_index(["title"]).index.difference(title_loc_df.index).empty
assert not full_aac_df["edition"].hasnans
assert title_loc_df.index.is_unique

In [ ]:
full_aac_df.set_index(["title"]).index.difference(title_loc_df.index).empty